# MediaPipe Pose+Hand Estimation

Setting up environment

In [2]:
import os
import glob
import pandas as pd #data wrangling
import csv #csv saving

curfolder = os.getcwd()
print("Current working directory is: " + curfolder)

# Projectdata ae one folder up
projectdata = os.path.dirname(curfolder) + '\\Input_Videos\\'  # check to the folder that you need
#projectdata = curfolder + "/ToTrack/"            # check to the folder that you need

vfiles = glob.glob(projectdata + "*\\*.mp4", recursive=True) #list all mp4 files in the folder
vfilesavi = glob.glob(projectdata + "*\\*.avi", recursive=True) #list all avi files in the folder
vfiles.extend(vfilesavi)  # combine both lists
outtputf_ts = curfolder + "\\OutputTimeSeries\\"
outputf_video = curfolder + "\\OutputVideos\\"

print("\n The following video(s) will be processed for masking: ")
print(vfiles)

Current working directory is: e:\FLESH_IteratedLearning\MotionTracking_complex

 The following video(s) will be processed for masking: 
['e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch2_g15_compr.mp4', 'e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch2_g16_compr.mp4', 'e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch2_g17_compr.mp4', 'e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch2_g18_compr.mp4', 'e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch2_g19_compr.mp4', 'e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch2_g20_compr.mp4', 'e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch3_g1_compr.mp4', 'e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch3_g2_compr.mp4', 'e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch3_g3_compr.mp4', 'e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch3_g4_compr.mp4', 'e:\\FLESH_IteratedLearning\\Input_Videos\\LNDW\\Donner_g_ch3_g5_compr.mp4', 'e:\\FLESH

Loading in MediaPipe

In [4]:
import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_holistic = mp.solutions.holistic

##################FUNCTIONS AND OTHER VARIABLES
#landmarks 33x that are used by Mediapipe (Blazepose)
markersbody = ['NOSE', 'LEFT_EYE_INNER', 'LEFT_EYE', 'LEFT_EYE_OUTER', 'RIGHT_EYE_OUTER', 'RIGHT_EYE', 'RIGHT_EYE_OUTER',
          'LEFT_EAR', 'RIGHT_EAR', 'MOUTH_LEFT', 'MOUTH_RIGHT', 'LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 
          'RIGHT_ELBOW', 'LEFT_WRIST', 'RIGHT_WRIST', 'LEFT_PINKY', 'RIGHT_PINKY', 'LEFT_INDEX', 'RIGHT_INDEX',
          'LEFT_THUMB', 'RIGHT_THUMB', 'LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE', 'LEFT_ANKLE', 'RIGHT_ANKLE',
          'LEFT_HEEL', 'RIGHT_HEEL', 'LEFT_FOOT_INDEX', 'RIGHT_FOOT_INDEX']

markershands = ['LEFT_WRIST', 'LEFT_THUMB_CMC', 'LEFT_THUMB_MCP', 'LEFT_THUMB_IP', 'LEFT_THUMB_TIP', 'LEFT_INDEX_FINGER_MCP',
              'LEFT_INDEX_FINGER_PIP', 'LEFT_INDEX_FINGER_DIP', 'LEFT_INDEX_FINGER_TIP', 'LEFT_MIDDLE_FINGER_MCP', 
               'LEFT_MIDDLE_FINGER_PIP', 'LEFT_MIDDLE_FINGER_DIP', 'LEFT_MIDDLE_FINGER_TIP', 'LEFT_RING_FINGER_MCP', 
               'LEFT_RING_FINGER_PIP', 'LEFT_RING_FINGER_DIP', 'LEFT_RING_FINGER_TIP', 'LEFT_PINKY_FINGER_MCP', 
               'LEFT_PINKY_FINGER_PIP', 'LEFT_PINKY_FINGER_DIP', 'LEFT_PINKY_FINGER_TIP',
              'RIGHT_WRIST', 'RIGHT_THUMB_CMC', 'RIGHT_THUMB_MCP', 'RIGHT_THUMB_IP', 'RIGHT_THUMB_TIP', 'RIGHT_INDEX_FINGER_MCP',
              'RIGHT_INDEX_FINGER_PIP', 'RIGHT_INDEX_FINGER_DIP', 'RIGHT_INDEX_FINGER_TIP', 'RIGHT_MIDDLE_FINGER_MCP', 
               'RIGHT_MIDDLE_FINGER_PIP', 'RIGHT_MIDDLE_FINGER_DIP', 'RIGHT_MIDDLE_FINGER_TIP', 'RIGHT_RING_FINGER_MCP', 
               'RIGHT_RING_FINGER_PIP', 'RIGHT_RING_FINGER_DIP', 'RIGHT_RING_FINGER_TIP', 'RIGHT_PINKY_FINGER_MCP', 
               'RIGHT_PINKY_FINGER_PIP', 'RIGHT_PINKY_FINGER_DIP', 'RIGHT_PINKY_FINGER_TIP']
facemarks = [str(x) for x in range(478)] #there are 478 points for the face mesh (see google holistic face mesh info for landmarks)

print("Note that we have the following number of pose keypoints for markers body")
print(len(markersbody))

print("\n Note that we have the following number of pose keypoints for markers hands")
print(len(markershands))

print("\n Note that we have the following number of pose keypoints for markers face")
print(len(facemarks ))

#set up the column names and objects for the time series data (add time as the first variable)
markerxyzbody = ['time']
markerxyzhands = ['time']
markerxyzface = ['time']

for mark in markersbody:
    for pos in ['X', 'Y', 'Z', 'visibility']: #for markers of the body you also have a visibility reliability score
        nm = pos + "_" + mark
        markerxyzbody.append(nm)
for mark in markershands:
    for pos in ['X', 'Y', 'Z']:
        nm = pos + "_" + mark
        markerxyzhands.append(nm)


Note that we have the following number of pose keypoints for markers body
33

 Note that we have the following number of pose keypoints for markers hands
42

 Note that we have the following number of pose keypoints for markers face
478


Needed functions (from EnvisionBox)

In [5]:
#check if there are numbers in a string
def num_there(s):
    return any(i.isdigit() for i in s)

#take some google classification object and convert it into a string
def makegoginto_str(gogobj):
    gogobj = str(gogobj).strip("[]")
    gogobj = gogobj.split("\n")
    return(gogobj[:-1]) #ignore last element as this has nothing

#make the stringifyd position traces into clean numerical values
def listpositions(newsamplemarks):
    newsamplemarks = makegoginto_str(newsamplemarks)
    tracking_p = []
    for value in newsamplemarks:
        if num_there(value):
            stripped = value.split(':', 1)[1]
            stripped = stripped.strip() #remove spaces in the string if present
            tracking_p.append(stripped) #add to this list  
    return(tracking_p)

# Motion tracking - Pose (pixels + meters) and Hand (pixels)

In [6]:
# We will now loop over all the videos that are present in the video file
for vidf in vfiles:
    print("We will now process video:")
    print(vidf)
    print("This is video number " + str(vfiles.index(vidf)) + " of " + str(len(vfiles)) + " videos in total")
    
    # Capture the video and check video settings
    videoname = vidf.split("\\")[-1]

    # If output video does exist, skip this file
    if os.path.exists(outputf_video + videoname):
        print("Output video file already exists, skipping this video.")
        continue

    capture = cv2.VideoCapture(vidf)  # load the video capture
    frameWidth = capture.get(cv2.CAP_PROP_FRAME_WIDTH)  # check frame width
    frameHeight = capture.get(cv2.CAP_PROP_FRAME_HEIGHT)  # check frame height
    samplerate = capture.get(cv2.CAP_PROP_FPS)  # fps = frames per second

    # Create an empty video file to project the pose tracking on
    fourcc = cv2.VideoWriter_fourcc(*'XVID')  # for different video formats you could use e.g., *'XVID', MP4V
    out = cv2.VideoWriter(outputf_video + videoname, fourcc, fps=samplerate, 
                          frameSize=(int(frameWidth), int(frameHeight)))
    
    # Initialize Mediapipe Holistic
    time = 0
    tsbody = [markerxyzbody]  # These are the time series objects starting with column names initialized above
    tshands = [markerxyzhands]
    tsbody_world = [markerxyzbody]  # For world landmarks (3D coordinates)


    with mp_holistic.Holistic(
        model_complexity=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
        static_image_mode=False,
        enable_segmentation=True
    ) as holistic:

        while capture.isOpened():
            ret, frame = capture.read()
            if not ret:
                break

            # Recolor image to RGB
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False

            # Make holistic detection
            results = holistic.process(image)

            # Recolor back to BGR
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            ### Pose landmarks
            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
                samplebody = listpositions(results.pose_landmarks)

                
                # Save pose world landmarks (3D coordinates in meters)
                if results.pose_world_landmarks:
                    samplebody_world = listpositions(results.pose_world_landmarks)
                    samplebody_world.insert(0, time)
                    tsbody_world.append(samplebody_world)


            else:
                samplebody = [np.nan for _ in range(len(markerxyzbody)-1)]
                samplebody.insert(0, time)
                tsbody.append(samplebody)

                # Append NaNs for world coordinates as well
                samplebody_world = [np.nan for x in range(len(markerxyzbody)-1)]
                samplebody_world.insert(0, time)
                tsbody_world.append(samplebody_world)

            ### Hand landmarks
            if results.left_hand_landmarks or results.right_hand_landmarks:
                for handLms in [results.left_hand_landmarks, results.right_hand_landmarks]:
                    if handLms:
                        mp_drawing.draw_landmarks(
                            image, handLms, mp_holistic.HAND_CONNECTIONS)
                        samplehands = listpositions(handLms)
                        samplehands.insert(0, time)
                        tshands.append(samplehands)
            

            else:
                samplehands = [np.nan for _ in range(len(markerxyzhands)-1)]
                samplehands.insert(0, time)
                tshands.append(samplehands)

            # Show and write output
            cv2.imshow('Mediapipe Feed', image)
            out.write(image)
            time += (1000 / samplerate)

            if cv2.waitKey(1) == 27:
                break
            if ret == False:  # if there are no more frames, break the loop
                break
            
    # Once done, de-initialize all processes
    out.release()
    capture.release()
    cv2.destroyAllWindows()

    # Save CSV data for body
    filebody = open(outtputf_ts + videoname + '_body.csv', 'w+', newline='')
    with filebody:
        write = csv.writer(filebody)
        write.writerows(tsbody)

     # Save world coordinates (in meters) to CSV for body, face, and hands
    filebody_world = open(outtputf_ts + videoname + '_body_world.csv', 'w+', newline='')
    with filebody_world:
        write = csv.writer(filebody_world)
        write.writerows(tsbody_world)

    # Save CSV data for hands
    filehands = open(outtputf_ts + videoname + '_hands_hol.csv', 'w+', newline='')
    with filehands:
        write = csv.writer(filehands)
        write.writerows(tshands)
    



We will now process video:
e:\FLESH_IteratedLearning\Input_Videos\LNDW\Donner_g_ch2_g15_compr.mp4
This is video number 0 of 440 videos in total
Output video file already exists, skipping this video.
We will now process video:
e:\FLESH_IteratedLearning\Input_Videos\LNDW\Donner_g_ch2_g16_compr.mp4
This is video number 1 of 440 videos in total
Output video file already exists, skipping this video.
We will now process video:
e:\FLESH_IteratedLearning\Input_Videos\LNDW\Donner_g_ch2_g17_compr.mp4
This is video number 2 of 440 videos in total
Output video file already exists, skipping this video.
We will now process video:
e:\FLESH_IteratedLearning\Input_Videos\LNDW\Donner_g_ch2_g18_compr.mp4
This is video number 3 of 440 videos in total
Output video file already exists, skipping this video.
We will now process video:
e:\FLESH_IteratedLearning\Input_Videos\LNDW\Donner_g_ch2_g19_compr.mp4
This is video number 4 of 440 videos in total
Output video file already exists, skipping this video.
We wi

# Motion tracking - Hand (meters)

from https://mediapipe.readthedocs.io/en/latest/solutions/hands.html

In [ ]:
import cv2
import mediapipe as mp


mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

# We will now loop over all the videos that are present in the video file
for vidf in vfiles:
    print("We will now process video:")
    print(vidf)
    print("This is video number " + str(vfiles.index(vidf)) + " of " + str(len(vfiles)) + " videos in total")
    
    # Capture the video and check video settings
    videoname = vidf.split("\\")[-1]
    if os.path.isfile(outputf_mask + videoname):
        print("The video file " + videoname + " already exists in the output folder. We will skip this video.")
        continue

    capture = cv2.VideoCapture(vidf)  # load the video capture

    # Initialize Mediapipe Holistic
    time = 0
    tshands = [markerxyzhands]
    tshands_world = [markerxyzhands]  # For normalized 3D hand landmarks


    with mp_hands.Hands(
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5) as hands:
      
      while capture.isOpened():
        success, image = capture.read()
        if not success:
          print("Ignoring empty camera frame.")
          # If loading a video, use 'break' instead of 'continue'.
          continue

        # To improve performance, optionally mark the image as not writeable to
        # pass by reference.
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image)

        # Draw the hand annotations on the image.
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        if results.multi_hand_landmarks:
          for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                image,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style())
            
            samplehands = listpositions(hand_landmarks)
            samplehands.insert(0, time)
            tshands.append(samplehands)
        
        if results.multi_hand_world_landmarks:
          for hand_world_landmarks in results.multi_hand_world_landmarks:
            samplehands_world = listpositions(hand_world_landmarks)
            samplehands_world.insert(0, time)
            tshands_world.append(samplehands_world)

        # Flip the image horizontally for a selfie-view display.
        cv2.imshow('MediaPipe Hands', cv2.flip(image, 1))
        if cv2.waitKey(5) & 0xFF == 27:
          break

    capture.release()
    cv2.destroyAllWindows()
